- https://www.kaggle.com/code/yosukeyama/rsna2025-32ch-img-infer-lb-0-69-share
のcodeの推論codeを参考にしつつ自分で学習codeを作成した

- データセットはkagglehubなどを使用してdownloadする。
- なお今回のコンペでは元のデータセットがかなり大きい(200GB〜)でkagglehubでdownloadするのは現実的ではなかったので、いったんkaggleの環境で.npyに変換(前処理)したものを使用していた。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Base directories (override via env vars for local/Colab)
DEFAULT_PROJECT_ROOT = "/content/drive/MyDrive/rsna2025"
DEFAULT_PREPROCESSED_DIR = "/content/preprocessed_6"

PROJECT_ROOT = Path(os.environ.get("RSNA_PROJECT_ROOT", DEFAULT_PROJECT_ROOT))
OUTPUT_ROOT = Path(os.environ.get("RSNA_OUTPUT_ROOT", PROJECT_ROOT.parent / "rsna2025_output"))
PREPROCESSED_DIR = Path(os.environ.get("RSNA_PREPROCESSED_DIR", DEFAULT_PREPROCESSED_DIR))

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from pathlib import Path
import os, shutil

# 入力元
DATASET_VERSIONS = ["2", "3"]
SRCS = [PROJECT_ROOT / v / "preprocessed_6" for v in DATASET_VERSIONS]

# 出力先（ここに2ツリーを統合）
DST = PREPROCESSED_DIR
DST.mkdir(parents=True, exist_ok=True)

# 設定
USE_SYMLINK = True   # True: symlink（速い/容量ゼロ）; False: 実ファイルをコピー
STRICT = False       # True: 相対パス衝突を検出したら例外で停止; False: 先勝ちで後発をスキップ

def iter_files(base: Path):
    for p in base.rglob("*"):
        if p.is_file():
            yield p, p.relative_to(base)

linked = copied = skipped = 0
dups = []

for src in SRCS:
    if not src.exists():
        print(f"⚠️ 入力が見つかりません: {src}")
        continue

    for f, rel in iter_files(src):
        out = DST / rel
        out.parent.mkdir(parents=True, exist_ok=True)

        if out.exists():
            # 同じ相対パスが既に作られている
            skipped += 1
            dups.append(str(rel))
            if STRICT:
                raise FileExistsError(f"相対パス衝突: {rel}（既に {out} が存在）")
            continue

        # 作成（まずはシンボリックリンク、ダメならコピー）
        try:
            if USE_SYMLINK:
                os.symlink(f, out)
                linked += 1
            else:
                shutil.copy2(f, out)
                copied += 1
        except OSError:
            # 環境によっては symlink 不可。その場合はコピーにフォールバック
            shutil.copy2(f, out)
            copied += 1

print("✅ 統合完了")
print(f"  symlink: {linked}, copied: {copied}, skipped(重複): {skipped}")
print(f"  出力先: {DST}")

# 重複の中身を確認したい場合は、下記をコメント解除
# for rel in dups[:20]:
#     print("DUP:", rel)


In [ ]:
!ls /content/

In [ ]:
!pip install pydicom -q

In [ ]:
# Environment setup and library imports
import os
import glob
import random
import warnings
import numpy as np
import pandas as pd
import cv2
import functools
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Optional,Dict
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score
import pydicom
import random
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from scipy import ndimage
import time
from tqdm import tqdm
import json
from sklearn.metrics import roc_auc_score
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    torch.cuda.empty_cache()
else:
    raise RuntimeError("CUDA is not available! This code requires GPU.")

In [ ]:
#日本の日にちと時間を取得する
from datetime import datetime, timedelta, timezone

# タイムゾーンの生成
JST = timezone(timedelta(hours=+9), 'JST')
now = datetime.now(JST)
print(now)

In [ ]:
class Config:
    """Training Configuration"""
    # Data paths
    csv: str = str(PROJECT_ROOT / "train.csv")  # Required
    dicom_root: str = str(PREPROCESSED_DIR)  # Required
    out_dir: str = str(OUTPUT_ROOT / f"models/output_{now}")
    cache_dir: str = None
    cache_mode: str = 'read_write'  # 'off', 'read', 'write', 'read_write'

    # Cross-validation
    n_folds: int = 5
    folds: str = None  # Comma-separated folds to train (None = all)

    # Training
    epochs: int = 5
    batch_size: int = 8
    valid_batch_size: int = 2
    num_workers: int = 2

    # Model
    size: int = 384
    in_chans: int = 1
    model_name: str = 'tf_efficientnetv2_s.in21k_ft_in1k'

    # Optimizer
    lr: float = 2e-4
    weight_decay: float = 1e-4
    cosine: bool = False

    # Training options
    amp: bool = True
    grad_accum_steps: int = 16
    seed: int = 42
    preprocessed_dir:str = str(PREPROCESSED_DIR)
    @classmethod
    def from_args(cls, args):
        """Create Config from argparse args"""
        config = cls()
        for key, value in vars(args).items():
            if hasattr(config, key):
                setattr(config, key, value)
        return config

    def __repr__(self):
        items = [f"{k}={v}" for k, v in self.__dict__.items() if not k.startswith('_')]
        return f"Config({', '.join(items)})"
# config = Config()


In [ ]:
# print(f"Train data shape: {train_df.shape}")
# print(f"Series mapping shape: {series_mapping_df.shape}")
# print(f"Localizers shape: {localizers_df.shape}")

# Define target columns

ID_COL = 'SeriesInstanceUID'
LABEL_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
    'Aneurysm Present',
]

### 前処理後の場合のDataset class

In [ ]:
from tqdm import tqdm
from pathlib import Path

# tqdmをpandasで使えるように設定
tqdm.pandas(desc="Checking files")

class RSNADataset(Dataset):
    """
    ファイルが存在しないサンプルを自動的にスキップする機能を持つDatasetクラス。
    """
    def __init__(
        self,
        df: pd.DataFrame,
        preprocessed_dir: str,
        transform: Optional[A.Compose] = None,
        metadata_features: Optional[Dict[str, np.ndarray]] = None,
        meta_dim: int = 0,
    ):
        self.preprocessed_dir = Path(preprocessed_dir)
        self.transform = transform
        self.metadata_features = metadata_features or {}
        self.meta_dim = meta_dim

        # --- ▼▼▼ ここからが修正・追加部分 ▼▼▼ ---

        print("Datasetを初期化中...")
        print(f"元のデータフレームのサイズ: {len(df)}")

        # 1. 各データに対応する.npyファイルのパスを生成
        df['npy_path'] = df[ID_COL].apply(lambda x: self.preprocessed_dir / f"{x}.npy")

        # 2. ファイルの存在をtqdmプログレスバー付きで確認し、新しい列 'file_exists' を作成
        df['file_exists'] = df['npy_path'].progress_apply(lambda p: p.exists())

        # 3. ファイルが存在するデータのみにデータフレームを絞り込む
        self.df = df[df['file_exists']].reset_index(drop=True)

        num_skipped = len(df) - len(self.df)
        if num_skipped > 0:
            print(f"警告: {num_skipped} 件のサンプルは、.npyファイルが見つからなかったためスキップされました。")

        print(f"利用可能なサンプル数（最終的なデータフレームのサイズ）: {len(self.df)}")

        # --- ▲▲▲ 修正・追加はここまで ▲▲▲ ---

    def __len__(self) -> int:
        # フィルタリング後のデータフレームの長さを返す
        return len(self.df)

    def _load_volume(self, npy_path: Path) -> np.ndarray:
        # __init__で存在確認済みなので、ここではエラーチェックなしで直接ロード
        volume = np.load(npy_path)
        if volume.ndim != 3:
            series_id = npy_path.stem
            raise ValueError(f"Expected 3D volume for {series_id}, got shape {volume.shape}")
        return volume.astype(np.float32)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        series_id = row[ID_COL]
        labels = row[LABEL_COLS].astype(float).values.astype(np.float32)

        # DataFrameに保存したパスを直接利用
        npy_path = row['npy_path']
        volume = self._load_volume(npy_path)

        image = volume.transpose(1, 2, 0)  # (H, W, D)
        if self.transform:
            transformed = self.transform(image=image)
            image = transformed['image']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1))

        if self.meta_dim > 0:
            meta = self.metadata_features.get(series_id)
            if meta is None:
                meta = np.zeros(self.meta_dim, dtype=np.float32)
            meta_tensor = torch.from_numpy(meta.astype(np.float32))
        else:
            meta_tensor = torch.zeros(self.meta_dim, dtype=torch.float32)

        target_tensor = torch.from_numpy(labels)
        return image,target_tensor

In [ ]:
#utils関数

#seed問題
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# def get_transforms(size:int):
#   return {
#       'train': A.Compose([
#           A.Normalize(),
#           ToTensorV2()
#       ]),
#       'valid': A.Compose([
#           A.Normalize(),
#           ToTensorV2()
#       ])
#   }

def get_transforms(size:int):
    common = [
        A.Resize(size, size),
        A.Normalize(),
        ToTensorV2()
    ]
    train_aug = A.Compose([
        A.OneOf([
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, border_mode=0, p=1.0),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), shear=(-5,5), rotate=(-10,10), fit_output=False, p=1.0),
        ], p=0.8),

        A.OneOf([
            A.ElasticTransform(alpha=50, sigma=6, alpha_affine=10, border_mode=0),
            A.GridDistortion(num_steps=5, distort_limit=0.2, border_mode=0),
            A.OpticalDistortion(distort_limit=0.2, shift_limit=0.05, border_mode=0),
        ], p=0.4),

        A.OneOf([
            A.RandomGamma(gamma_limit=(80,120)),
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1),
        ], p=0.5),

        A.OneOf([
            A.GaussNoise(var_limit=(5.0, 25.0)),
            A.MultiplicativeNoise(multiplier=(0.9,1.1), per_channel=False),
            A.MotionBlur(blur_limit=3),
            A.GaussianBlur(blur_limit=3),
        ], p=0.4),

        A.CoarseDropout(max_holes=4, max_height=int(0.1*size), max_width=int(0.1*size), p=0.3),

        # 左右反転はモダリティ/タスク次第でオンに
        # A.HorizontalFlip(p=0.0 if chest_xray else 0.5),
    ] + common)

    valid_aug = A.Compose(common)
    return {'train': train_aug, 'valid': valid_aug}


In [ ]:
class MLPHead(nn.Module):
    """4層MLP with Residual Connection"""
    def __init__(self, lstm_out_dim: int, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.norm = nn.LayerNorm(lstm_out_dim)
        self.fc1 = nn.Linear(lstm_out_dim, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.gelu = nn.GELU()

    def forward(self, x):
        x = self.norm(x)
        x = self.fc1(x)
        identity = x  # Residual connection
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = x + identity  # Add residual
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.fc4(x)
        return x

In [ ]:
# pip install timm torch torchvision
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import timm
from typing import Optional, Tuple

class TimmPerSliceLSTM(nn.Module):
    """
    2D timm バックボーンで各スライスを 1280-d に埋め込み → LSTM で T 方向を集約 → Head で分類
    入力は以下のどれでもOK（自動で [B,T,C,H,W] に正規化）:
      - [D, H, W]                         # (32,384,384)
      - [B, D, H, W]                      # バッチ化した3D
      - [B, T, C, H, W] / [B, C, D, H, W] # 本来の5D
    既定: C=1（SamplesPerPixel=1想定）
    """
    def __init__(
        self,
        model_name: str,
        num_classes: int,
        in_chans: int,
        lstm_hidden: int = 512,
        lstm_layers: int = 1,
        bidirectional: bool = True,
        head_hidden: int = 256,
        dropout: float = 0.1,
        agg: str = "mean",            # "mean" | "last"
        cnn_chunk_size: Optional[int] = None,
        pretrained: bool = True,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            num_classes=0,
            in_chans=in_chans,
            pretrained=pretrained,
            global_pool="avg",
        )
        self.feat_dim = getattr(self.backbone, "num_features", 1280)
        self.cnn_chunk_size = cnn_chunk_size

        self.lstm = nn.LSTM(
            input_size=self.feat_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )
        lstm_out_dim = lstm_hidden * (2 if bidirectional else 1)

        # self.head = nn.Sequential(
        #     nn.LayerNorm(lstm_out_dim),
        #     nn.Dropout(dropout),
        #     nn.Linear(lstm_out_dim, num_classes),
        # )

        # 深いMLPヘッド（Residual付き）
        self.head = MLPHead(lstm_out_dim, num_classes, dropout=0.2)

        assert agg in ("mean", "last")
        self.agg = agg
        self.in_chans = in_chans

    # ---- 旧モデルのLR分離を踏襲（backbone / lstm / head）----
    def param_groups(self, lr_backbone: float, lr_lstm: float, lr_head: float, weight_decay: float = 0.05):
        return [
            {"params": self.backbone.parameters(), "lr": lr_backbone, "weight_decay": weight_decay},
            {"params": self.lstm.parameters(),     "lr": lr_lstm,     "weight_decay": weight_decay},
            {"params": self.head.parameters(),     "lr": lr_head,     "weight_decay": weight_decay},
        ]

    # ---- どの形で来ても [B,T,C,H,W] -> [B*T,C,H,W] にそろえる ----
    def _normalize_to_BTCHW(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, int, int, int]:
        """
        returns:
            x_flat: [B*T, C, H, W]
            lengths: [B]   # 各サンプルT
            B, T, C
        """
        assert x.ndim in (3, 4, 5), f"input ndim must be 3/4/5, got {x.ndim}"

        if x.ndim == 3:
            # [D, H, W] -> [1, D, 1, H, W]
            x = x.unsqueeze(0).unsqueeze(2)

        elif x.ndim == 4:
            # ほぼ確実に [B, D, H, W] とみなして [B, D, 1, H, W] へ
            # （もし [B,C,H,W] を渡したいなら per-sliceの前提に合わないので個別対応推奨）
            x = x.unsqueeze(2)

        # ここまでで [B, T(=D), C, H, W] or [B, C, D, H, W]
        if x.shape[1] == self.in_chans and x.shape[2] != self.in_chans:
            # [B, C, D, H, W] -> [B, D, C, H, W]
            x = x.permute(0, 2, 1, 3, 4).contiguous()

        B, T, C, H, W = x.shape
        if C != self.in_chans:
            raise AssertionError(f"in_chans mismatch: expected {self.in_chans}, got {C}")

        lengths = torch.full((B,), T, dtype=torch.long, device=x.device)
        x_flat = x.view(B * T, C, H, W)
        return x_flat, lengths, B, T, C

    def _extract_feats(self, x_flat: torch.Tensor, B: int, T: int) -> torch.Tensor:
        if self.cnn_chunk_size is None:
            f = self.backbone(x_flat)
        else:
            outs = []
            for s in range(0, x_flat.size(0), self.cnn_chunk_size):
                e = min(s + self.cnn_chunk_size, x_flat.size(0))
                outs.append(self.backbone(x_flat[s:e]))
            f = torch.cat(outs, dim=0)
        return f.view(B, T, -1)

    def _aggregate_T(self, seq_out: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        B, T, D = seq_out.shape
        if self.agg == "mean":
            idx = torch.arange(T, device=seq_out.device)[None, :]
            mask = (idx < lengths[:, None]).unsqueeze(-1)  # [B,T,1]
            summed = (seq_out * mask).sum(1)
            return summed / lengths.clamp_min(1).unsqueeze(1)
        # last
        gather_idx = (lengths - 1).clamp_min(0).view(B, 1, 1).expand(B, 1, D)
        return seq_out.gather(dim=1, index=gather_idx).squeeze(1)

    def forward(self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None):
        x_flat, auto_lengths, B, T, _ = self._normalize_to_BTCHW(x)
        feats = self._extract_feats(x_flat, B, T)  # [B, T, feat_dim]

        if lengths is None:
            lengths = auto_lengths
        packed = pack_padded_sequence(feats, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        seq_out, _ = pad_packed_sequence(packed_out, batch_first=True)  # [B, T, lstm_out]

        study_feat = self._aggregate_T(seq_out, lengths)                # [B, lstm_out]
        logits = self.head(study_feat)                                  # [B, num_classes]
        return logits


def build_model(
    model_name: str,
    num_classes: int,
    in_chans: int,
    lstm_hidden: int = 512,
    lstm_layers: int = 1,
    bidirectional: bool = True,
    head_hidden: int = 256,
    dropout: float = 0.1,
    agg: str = "mean",
    cnn_chunk_size: Optional[int] = None,
    pretrained: bool = True,
) -> nn.Module:
    return TimmPerSliceLSTM(
        model_name=model_name,
        num_classes=num_classes,
        in_chans=in_chans,
        lstm_hidden=lstm_hidden,
        lstm_layers=lstm_layers,
        bidirectional=bidirectional,
        head_hidden=head_hidden,
        dropout=dropout,
        agg=agg,
        cnn_chunk_size=cnn_chunk_size,
        pretrained=pretrained,
    )


In [ ]:
#train 1 epochの関数
"""
流れは、model.train()をして
1 順伝播 model
2 逆伝播 loss.backward()
3 パラメータ更新 optimizer.step
4 勾配リセット optimizer.zero_grad()
AMPと勾配累積などを使用している
"""
#勾配累積をする時は、予めその累積分だけをlossで割ってから足していく

def train_one_epoch(model, loader, criterion, optimizer, device, scaler:Optional[GradScaler]=None, use_amp:bool=True, grad_accum_steps:int = 1):
  model.train()
  running_loss =0.0
  optimizer.zero_grad(set_to_none=True)

  pbar = tqdm(enumerate(loader), total=len(loader), desc='Training', leave=False)

  for step, (images, targets) in pbar:
    #modelと同じdeviceに移動する
    images = images.to(device)
    targets = targets.to(device)
    #自動混合精度
    with autocast(enabled=use_amp): #自動的にbloat16で計算する
      logits = model(images) #順伝播
      loss = criterion(logits, targets)
      loss = loss / grad_accum_steps #勾配累積分だけ予め割っておく
    if scaler is not None and use_amp: #逆伝播
      scaler.scale(loss).backward() #bf16用(勾配が小さいのでスケールが必要な場合がある)
    else:
      loss.backward()
    # 勾配累積 grad_accum_stepsごとにパラメータの更新をする
    if (step+1) % grad_accum_steps == 0:
      if scaler is not None and use_amp:
        scaler.step(optimizer)
        scaler.update()
      else:
        optimizer.step()
      optimizer.zero_grad(set_to_none=True)
    running_loss += loss.item() * grad_accum_steps
    pbar.set_postfix({
        "loss": f"{running_loss / (step+1):.4f}"
    })
  return running_loss / max(1, len(loader)) #batch数で割る


In [ ]:
#validation用
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, use_amp:bool=True):
  model.eval()

  running_loss = 0.0
  all_preds =[]
  all_targets = []
  pbar = tqdm(enumerate(loader), total=len(loader), desc='Validation', leave=False)

  for step,(images, targets) in pbar:
    images = images.to(device)
    targets = targets.to(device)
    with autocast(enabled=use_amp):
      logits = model(images)
      loss = criterion(logits, targets)
    probs = torch.sigmoid(logits)
    running_loss += loss.item()
    all_preds.append(probs.cpu().numpy())
    all_targets.append(targets.cpu().numpy())

    pbar.set_postfix({
        "loss": f"{running_loss / (step+1):.4f}",
    })
  all_preds = np.concatenate(all_preds, axis=0)
  all_targets = np.concatenate(all_targets,axis=0)
  metrics = calculate_competition_metric(all_targets,all_preds)
  avg_loss = running_loss / max(1, len(loader))

  return avg_loss, metrics

In [ ]:
#chackpoint保持

def save_checkpoint(path:Path, model: nn.Module, epoch: int, best_score: float, optimizer, scaler: Optional[GradScaler] = None):
  path.parent.mkdir(parents=True, exist_ok=True)
  ckpt = {
      "model":model.state_dict(),
      "epoch":epoch,
      "best_score":best_score,
      "optimizer":optimizer.state_dict() if optimizer is not None else None,
      "scaler": scaler.state_dict() if scaler is not None else None,
  }
  torch.save(ckpt, path)

In [ ]:
def calculate_competition_metric(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_cols: Optional[List[str]] = None
) -> Dict[str, float]:
    """
    RSNA 2025 Weighted Macro-AUC:
      - 最後のクラス（Aneurysm Present）: weight = 13
      - それ以外: weight = 1
      - 正規化した重みで加重平均
    """
    assert y_true.shape == y_pred.shape, "y_true と y_pred の shape は一致している必要があります"
    n_classes = y_true.shape[1]

    # 重みベクトル [1, 1, ..., 1, 13]
    weights = np.ones(n_classes, dtype=float)
    weights[-1] = 13.0
    weights_norm = weights / weights.sum()  # 例: クラス数14なら合計26で割る

    # クラスごとの AUC（正例/負例の両方が存在しない場合は 0.5 を採用）
    indiv_auc = np.empty(n_classes, dtype=float)
    for i in range(n_classes):
        y = y_true[:, i]
        p = y_pred[:, i]
        if len(np.unique(y)) < 2:
            indiv_auc[i] = 0.5
        else:
            indiv_auc[i] = float(roc_auc_score(y, p))

    weighted_auc = float(np.sum(indiv_auc * weights_norm))
    mean_auc = float(np.mean(indiv_auc))

    # 出力（列名があればそれを使う）
    cols = label_cols if label_cols is not None else [f"class_{i}" for i in range(n_classes)]
    results: Dict[str, float] = {cols[i]: float(indiv_auc[i]) for i in range(n_classes)}
    results["mean_auc"] = mean_auc
    results["weighted_auc"] = weighted_auc
    return results

In [ ]:
def run_folds(
    fold: int,
    df:pd.DataFrame,
    config: Config,
    device,
    ):
  print(f"===================Fold{fold}===================")
  #cv
  trn_df = df[df['fold'] !=fold].reset_index(drop=True)
  val_df = df[df["fold"] == fold].reset_index(drop=True)
  print(f"Train: {len(trn_df)} Valid: {len(val_df)}")

  transforms = get_transforms(config.size)
  #preprocessor = DICOMPreprocessorKaggle(target_shape=(config.in_chans, config.size, config.size))

  #train_ds = RSNADataset(
  #     trn_df, config.dicom_root, preprocessor, transforms['train'],
  #     cache_dir=config.cache_dir, cache_mode=config.cache_mode,
  # )
  #val_ds = RSNADataset(
  #     val_df, config.dicom_root, preprocessor, transforms['valid'],
  #     cache_dir=config.cache_dir, cache_mode="read" if config.cache_mode != "off" else "off",
  # )
  #前処理済みを使用する場合にはこちらを使用する
  train_ds = RSNADataset(trn_df, config.preprocessed_dir, transforms['train'])
  val_ds = RSNADataset(val_df, config.preprocessed_dir, transforms['valid'])

  train_loader = DataLoader(
      train_ds, batch_size=config.batch_size, shuffle=True,
      num_workers=config.num_workers, pin_memory=True, drop_last=True,
  )
  val_loader = DataLoader(
      val_ds, batch_size=config.valid_batch_size, shuffle=False,
      num_workers=config.num_workers, pin_memory=True,
  )

  model = build_model(config.model_name, num_classes=len(LABEL_COLS), in_chans=config.in_chans).to(device)

  optimizer = AdamW(model.param_groups(
    lr_backbone=1e-5,   # 小さめ（微調整）
    lr_lstm=1e-3,
    lr_head=1e-3,       # 大きめ（ランダム初期化のMLP）
    weight_decay=0.05,
))

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, config.epochs)) if config.cosine else None
  criterion = nn.BCEWithLogitsLoss()
  scaler = GradScaler(enabled=config.amp) if config.amp else None

  #valで追跡するのはweighted ACUの値
  best_score = 0.0
  best_path = Path(config.out_dir) / f"{config.model_name}_fold{fold}_best.pth"
  epoch_pbar = tqdm(range(1, config.epochs + 1), desc=f'Fold {fold}')


  for epoch in epoch_pbar:
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, config.amp, config.grad_accum_steps)
    val_loss,metrics = validate_one_epoch(model, val_loader, criterion, device, config.amp)

    if scheduler is not None:
      scheduler.step()
    dt = time.time() - t0
    print(f"Epoch {epoch:02d} | train_loss  {tr_loss:.4f} | valid_loss {val_loss:.4f} | time {dt:.1f}s")
    print(f"  Weighted Score: {metrics['weighted_auc']:.4f} | Mean AUC: {metrics['mean_auc']:.4f}")

    if metrics['weighted_auc'] > best_score:
        best_score = metrics["weighted_auc"]
        save_checkpoint(best_path, model, epoch, best_score, optimizer, scaler)
        print(f" -> SAVED best model score{best_score:.4f}　")

    #epoch終了ごとにmemoryをclean up
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
    gc.collect()
  return best_score




In [ ]:
def add_folds(df:pd.DataFrame, n_splits:int, seed:int) -> pd.DataFrame:
  from sklearn.model_selection import StratifiedKFold

  if "fold" in df.columns:
    return df

  #numpy配列に変換する(scikit-learnは内部でNumpy配列を期待する)
  labels = df["Aneurysm Present"].astype(int).values
  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
  df = df.copy()
  df['fold'] = -1
  #検証用の方にだけfold番号をつける
  for fold, (_, val_idx) in enumerate(skf.split(df, labels)):
    df.loc[val_idx, 'fold'] = fold
  return df

In [ ]:
def main():
  # args = parse_args()
  config = Config()

  seed_everything(config.seed)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  print(f"device{device}")
  print(f"Config {config}")

  #load csv
  df = pd.read_csv(config.csv)
  #df = df.head(50)
  #validate
  missing_cols = [c for c in [ID_COL] + LABEL_COLS if c not in df.columns]
  if missing_cols :
    raise ValueError(f"Missing columns: {missing_cols}")

  #層化fold
  df = add_folds(df, n_splits=config.n_folds, seed=config.seed)

  if config.folds is not None:
    selected = [int(x) for x in config.folds.split(',') if len(x) >0]
  else:
    selected = list(range(config.n_folds))
  print(f"Folds to train : {selected}")

  os.makedirs(config.out_dir, exist_ok=True)
  fold_scores = {}

  for f in selected:
    score = run_folds(f, df, config, device)
    fold_scores[f] = score



  print("\n" + "="*60)
  print("Training Finished!")
  print("="*60)
  print(f"Average Weighted Score: {np.mean(list(fold_scores.values())):.4f}")
  print(f"Fold Scores: {json.dumps(fold_scores, indent=2)}")


In [ ]:
# 出力だけをファイルに同時保存する（表示もそのまま） logを吐き出す
import sys, io, datetime, os

# Google Driveに保存する例（ローカルなら適宜変更）
log_dir = OUTPUT_ROOT / "logs"
os.makedirs(log_dir, exist_ok=True)
log_path = log_dir / f"run_{now}.log"

class _Tee(io.TextIOBase):
    def __init__(self, *targets): self.targets = targets
    def write(self, s):
        for t in self.targets:
            t.write(s); t.flush()
        return len(s)
    def flush(self):
        for t in self.targets: t.flush()

# もとを退避して、以降の全セル出力を“複製”保存
_orig_stdout, _orig_stderr = sys.stdout, sys.stderr
_log_file = open(log_path, "w", encoding="utf-8", buffering=1)  # 行ごとに即書き出し
sys.stdout = _Tee(sys.stdout, _log_file)
sys.stderr = _Tee(sys.stderr, _log_file)

print("→ logging to:", log_path)


In [ ]:
if __name__ == '__main__':
  main()

In [ ]:
# ログを終了し、標準出力を元に戻す
sys.stdout, sys.stderr = _orig_stdout, _orig_stderr
_log_file.close()
print("log saved to:", log_path)

In [ ]:
import numpy as np
example_path = PROJECT_ROOT / "preprocessed_2" / "example_series.npy"  # replace with any sample id file
arr = np.load(example_path)
print(arr.shape, arr.dtype)


In [ ]:
from IPython.display import Javascript
display(Javascript('google.colab.kernel.disconnect()'))
#これで学習が終わると自動でColabランタイムが切断される

<IPython.core.display.Javascript object>